# RSNA Knee — Submission Notebook
Attach: competition data + the `rsna-knee-weights-r2` dataset. Internet OFF, GPU ON.
Produces `submission.csv` via the packaged `predict.py` (in-memory DICOM decode, 8-checkpoint ensemble).

In [ ]:
import sys, shutil, subprocess
from pathlib import Path

IN = Path("/kaggle/input")
print("inputs:", [str(p) for p in IN.rglob("*") if p.is_dir() and len(p.relative_to(IN).parts) <= 3][:20])

def find_dir(name_prefix):
    cands = [p for p in IN.rglob("*") if p.is_dir() and p.name.startswith(name_prefix)
             and len(p.relative_to(IN).parts) <= 3]
    return sorted(cands, key=lambda p: len(p.parts))[0] if cands else None

COMP = find_dir("rsna-knee-abnormality-detection")
WEIGHTS = find_dir("rsna-knee-weights")
print("competition:", COMP, "| weights:", WEIGHTS)
assert COMP is not None and WEIGHTS is not None, "input mounts missing"
sys.path.insert(0, str(WEIGHTS/"src"))
import pydicom, timm, torch, cv2
print("pydicom", pydicom.__version__, "| timm", timm.__version__, "| torch", torch.__version__)

In [ ]:
# checkpoints -> the fold-dir layout predict.py expects
ck_root = Path("/kaggle/working/ckpts")
for p in (WEIGHTS/"ckpts").glob("*.pt"):
    stem = p.stem                      # e.g. r2_v1_fold0
    d = ck_root/stem.rsplit("_", 1)[0]/stem.rsplit("_", 1)[1]
    d.mkdir(parents=True, exist_ok=True)
    if not (d/"best.pt").exists():
        shutil.copy(p, d/"best.pt")
dirs = sorted(str(x) for x in ck_root.iterdir())
print(dirs)

In [ ]:
cmd = [sys.executable, str(WEIGHTS/"src/predict.py"),
       "--test-csv", str(COMP/"test.csv"),
       "--series-csv", str(COMP/"test_series.csv"),
       "--dicom-root", str(COMP/"test_series"),
       "--ckpt-dirs", *dirs,
       "--model-src", str(WEIGHTS/"src"),
       "--out", "/kaggle/working/submission.csv"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
assert list(sub.columns) == ["StudyInstanceUID","ACL","MCL","Medial Meniscus","Lateral Meniscus",
    "Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
assert sub.notna().all().all() and ((sub.iloc[:, 1:] >= 0) & (sub.iloc[:, 1:] <= 1)).all().all()
print(sub.shape); sub.head(3)